# Numerical Analysis Project 1
##  Authors: Bopeng Zhang (bz292), Daniel Chuang (dc863), Tianrui Wang (tw559)

In [160]:
# Project Setup
import numpy as np
import random
from typing import *
import matplotlib.pyplot as plt

## Goal 1:
Implement the Householder scheme for computing QR factorizations. Your QR factorization routine should take in a $n \times k$ matrix $B$ with $k\leq n$ and returns a $n\times k$ matrix $Q$ with orthonormal columns (or sufficient information to be able to apply it to a vector, e.g., the Householder vectors), and a $k\times k$ upper triangular matrix $R.$ Please demonstrate that your algorithm behaves as expected (this means both that you are getting a valid QR factorization as output and that it achieves the desired computational scaling). Demonstrate that your implementation achieves the expected scaling and explain how you tested your implementation.

In [82]:
# Translated from David Bindel's CS 4220 Notes
# Source: https://www.cs.cornell.edu/courses/cs4220/2023sp/lec/2023-02-22.html

def QR_Householder_helper(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    m, n = A.shape
    A = A.copy() # 
    tau = np.zeros(n)

    for j in range(n):
        normx = np.linalg.norm(A[j:, j])
        if normx < 1e-14: # Avoid div by 0 if normx very small
            continue
        s = -np.sign(A[j, j])
        s = -1 if A[j, j] == 0 else s
        u1 = A[j, j] - s * normx
        w = A[j:, j] / u1
        w[0] = 1.0
        if j + 1 < m:
            A[j+1:, j] = w[1:]
        A[j, j] = s * normx
        tau[j] = -s * u1 / normx
        if j + 1 < n: # Ensure we have columns to update
            # First calculate w' * A[j:, j+1:]
            wTA = w @ A[j:, j+1:] # This is a row vector (1 x n-j)
            # Then calculate w * (w'*A[j:, j+1:])
            # w is a column vector, wTA is a row vector
            # Outer product to get a matrix of the right shape
            update = np.outer(w, wTA)
            # Apply the update
            A[j:, j+1:] -= tau[j] * update
    return A, tau

def form_Q(A_mod: np.ndarray, tau: np.ndarray) -> np.ndarray:
    """
    Form the Q matrix from the Householder reflectors stored in A_mod and tau.
    
    Args:
        A_mod (ndarray): Mod. A matrix that has the Householder vectors
        tau (ndarray): Scaling factors for the Householder reflections
        
    Output:
        Q (ndarray): orthog Q
    """
    m, n = A_mod.shape
    Q = np.eye(m, n)  # This creates a m×n matrix with ones on the diagonal
    
    for j in range(n-1, -1, -1):
        # Extract the Householder vector
        w = np.zeros(m - j)
        w[0] = 1.0
        if j + 1 < m:
            w[1:] = A_mod[j+1:, j]
        
        # Apply the Householder reflection to Q
        wTQ = w @ Q[j:, :]
        update = np.outer(w, wTQ)
        Q[j:, :] -= tau[j] * update
    
    return Q

def extract_R(A_mod: np.ndarray) -> np.ndarray:
    """
    Extract the R matrix from the modified A matrix.
    
    Args:
        A_mod (ndarray): Modified A matrix containing the Householder vectors
    
    Output:
        R (ndarray): The upper triangular matrix R
    """
    m, n = A_mod.shape
    R = np.zeros((n, n))
    
    for i in range(n):
        R[i, i:] = A_mod[i, i:]
    
    return R

def QR_Householder(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute a reduced QR factorization for matrix A via 
    
    Args:
        A (ndarray): n x k, Input matrix with k <= n
    
    Output:
        Q (ndarray): n x k, orthog
        R (ndarray): k x k, upper triangular
    """
    A_mod, tau = QR_Householder_helper(A)
    Q = form_Q(A_mod, tau)
    R = extract_R(A_mod)
    return Q, R

In [83]:
for k in range(1, 10):
  for n in range(k, 10):
    A = np.random.rand(5, 3)
    Q_hh, R_hh = QR_Householder(A)
    Q, R = np.linalg.qr(A, mode="reduced")
    assert(np.allclose(Q_hh @ R_hh, A))
    assert(np.allclose(Q_hh, Q))
    assert(np.allclose(R_hh, R))
print("All test cases passed")

All test cases passed


## Goal 2:
Using your $QR$ factorization, implement Algorithm 1. This is written as a matrix least squares problem, but think about the Frobenius norm and how you might be able to split it up into problems we know how to solve. **Please explain how you do this in the project report.** Practically, it is possible to then block some of these operations together for efficiency; you may take advantage of this if you wish.

In [162]:
# We will solve this by four steps
# 1. Backsolver for triangular
# 2. Matrix case
# 3. Implement Alternating LS

# https://www.cs.cornell.edu/courses/cs4220/2023sp/lec/2023-02-08.pdf
def backward_subst(R, Qt_b):
  """Backward substitution

  Args:
      R (ndarray):
      Qt_b (ndarray):

  Output:
      x (ndarray): 

  """
  n = Qt_b.shape[0]
  x = np.copy(Qt_b)
  for i in range(n-1, -1, -1):
      x[i] = (x[i] - np.sum(R[i, i+1:n] * x[i+1:n])) / R[i, i]
  return x

def vector_ls(A, b):
  Q, R = QR_Householder(A)
  return backward_subst(R, Q.T @ b)

def matrix_ls(A, B):
    """
    Matrix Least Squares Solver via QR factorization.
    Solves the problem: min ||A·X - B||_F
    
    Args:
        A (ndarray): m×n matrix, m >= n
        B (ndarray): m×p matrix
        
    Returns:
        X (ndarray): n×p solution matrix
    """
    m, n = A.shape
    p = B.shape[1]
    
    Q, R = QR_Householder(A)
    
    X = np.zeros((n, p))
    
    for j in range(p):
        y = Q.T @ B[:, j]
        X[:, j] = backward_subst(R, y[:n])
    return X

def alternating_ls(A, k, num_iterations=9999, tol=1e-2):
    n1, n2 = A.shape
    W = np.random.randn(n1,k)
    Zt = np.random.randn(k,n2)

    prev_error = np.linalg.norm(A - W @ Zt, 'fro')
    for i in range(num_iterations):
        Zt = matrix_ls(W, A)
        W = matrix_ls(Zt.T, A.T).T

        current_error = np.linalg.norm(A - W @ Zt, 'fro')
        print(current_error)
        error_change = abs(prev_error - current_error)
        if error_change < tol:
            print(f"Converged after {i+1} iterations")
            break
            
        prev_error = current_error

    return W @ Zt

In [163]:
# Test backward_sub
for k in range(1, 10):
    for n in range(k, 10):
        U = np.triu(np.random.rand(n, n))
        for i in range(n):
            if abs(U[i, i]) < 1e-10:
                U[i, i] = 1.0
        d = np.random.rand(n)
        x_our = backward_subst(U, d)
        x_numpy = np.linalg.solve(U, d)
        assert np.allclose(x_our, x_numpy)
        assert np.allclose(U @ x_our, d)

print("All backward substitution tests passed!")

# Test vector LS
for n in range(2, 20):
    m = random.randint(n, n + 3)
    A = np.random.rand(m, n)
    b = np.random.rand(m)
    x_our = vector_ls(A, b)
    x_numpy = np.linalg.lstsq(A, b)[0]
    assert(np.allclose(x_our, x_numpy))
  
print("All vector ls tests passed!")

def test_matrix_ls(verbose = True):
    """
    Test the matrix_ls function by:
      - Generating a random system A * X ≈ B
      - Solving using matrix_ls(A, B)
      - Comparing the result against np.linalg.lstsq
    """
    np.random.seed(42)

    # Dimensions: m >= n
    m, n = 8, 5   # A is (m × n)
    p = 3         # B is (m × p)

    # Create a "true" solution X_true
    X_true = np.random.randn(n, p)

    # Generate A and form B = A * X_true + noise
    A = np.random.randn(m, n)
    noise = 0.01 * np.random.randn(m, p)
    B = A @ X_true + noise

    # Solve using matrix_ls function
    X_est = matrix_ls(A, B)

    # Solve using NumPy's built-in lstsq for reference
    X_lstsq, residuals, rank, s = np.linalg.lstsq(A, B, rcond=None)

    # Calculate error metrics
    err_true_est = np.linalg.norm(X_est - X_true, 'fro')
    err_true_lstsq = np.linalg.norm(X_lstsq - X_true, 'fro')
    err_est_lstsq = np.linalg.norm(X_est - X_lstsq, 'fro')
    residual_est = np.linalg.norm(A @ X_est - B, 'fro')
    residual_lstsq = np.linalg.norm(A @ X_lstsq - B, 'fro')

    if verbose:
      print("TEST: matrix_ls vs. np.linalg.lstsq")
      print(f"Shape of A: {A.shape}, B: {B.shape}, X_est: {X_est.shape}")
      print(f"Error ||X_est - X_true||_F       = {err_true_est:.6e}")
      print(f"Error ||X_lstsq - X_true||_F     = {err_true_lstsq:.6e}")
      print(f"Difference ||X_est - X_lstsq||_F = {err_est_lstsq:.6e}")
      print(f"Residual ||A*X_est - B||_F       = {residual_est:.6e}")
      print(f"Residual ||A*X_lstsq - B||_F     = {residual_lstsq:.6e}")
      print("-" * 60)
    assert(np.allclose(X_est, X_lstsq))

for i in range(100):
  test_matrix_ls(verbose = False)

print("All matrix ls tests passed!")

# Test Alternating LS

def test_QR_factorization():
    """
    Test the Householder QR factorization by:
    1. Verifying that Q is orthogonal (Q^T Q ≈ I)
    2. Verifying that A = QR
    3. Comparing with NumPy's QR factorization
    4. Testing computational scaling with different matrix sizes
    """
    import time
    import numpy as np
    
    print("Testing QR Factorization")
    print("=======================")
    
    matrix_sizes = [(10, 5), (20, 10), (50, 25), (100, 50)]
    
    for m, n in matrix_sizes:
        print(f"\nTesting {m}×{n} matrix:")
        
        np.random.seed(42)  
        A = np.random.randn(m, n)
        
        start_time = time.time()
        Q, R = QR_Householder(A)
        our_time = time.time() - start_time
        
        start_time = time.time()
        Q_np, R_np = np.linalg.qr(A, mode='reduced')
        np_time = time.time() - start_time
        
        ortho_error = np.linalg.norm(Q.T @ Q - np.eye(n), 'fro')
        
        reconstruction_error = np.linalg.norm(A - Q @ R, 'fro') / np.linalg.norm(A, 'fro')
        
        np_diff = np.linalg.norm(Q @ R - Q_np @ R_np, 'fro') / np.linalg.norm(A, 'fro')
        
        print(f"  Orthogonality error: {ortho_error:.2e}")
        print(f"  Reconstruction error: {reconstruction_error:.2e}")
        print(f"  Difference from NumPy: {np_diff:.2e}")
        print(f"  Time for our implementation: {our_time:.5f} seconds")
        print(f"  Time for NumPy implementation: {np_time:.5f} seconds")
        print(f"  Speed ratio (NumPy/Ours): {np_time/our_time:.2f}x")
    
    # Test edge cases
    print("\nTesting edge cases:")
    
    # 1. Matrix with a zero column
    A = np.random.randn(5, 3)
    A[:, 1] = 0
    Q, R = QR_Householder(A)
    reconstruction_error = np.linalg.norm(A - Q @ R, 'fro') / np.linalg.norm(A, 'fro')
    print(f"  Matrix with zero column - Reconstruction error: {reconstruction_error:.2e}")
    
    # 2. Matrix with identical columns
    A = np.random.randn(5, 3)
    A[:, 2] = A[:, 0]
    Q, R = QR_Householder(A)
    reconstruction_error = np.linalg.norm(A - Q @ R, 'fro') / np.linalg.norm(A, 'fro')
    print(f"  Matrix with identical columns - Reconstruction error: {reconstruction_error:.2e}")
    
    # 3. Tall skinny matrix
    A = np.random.randn(100, 2)
    Q, R = QR_Householder(A)
    reconstruction_error = np.linalg.norm(A - Q @ R, 'fro') / np.linalg.norm(A, 'fro')
    print(f"  Tall skinny matrix - Reconstruction error: {reconstruction_error:.2e}")
    assert(np.allclose(X_est, X_lstsq))


# for i in range(100):
#   test_alternating_ls(verbose = False)
# print("All alternating ls tests passed!")


All backward substitution tests passed!
All vector ls tests passed!
All matrix ls tests passed!


/var/folders/rp/nchg416j6b37gngjnnhbxrsc0000gp/T/ipykernel_74766/313239365.py:22: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  x_numpy = np.linalg.lstsq(A, b)[0]


## Goal 3:
Using your implementation, load the file Cornell.csv which has an image stored in the matrix $C$ and try to compute the best rank 75 approximation of the image. Compare this with the result of the best rank 75 image from the SVD (you can use a built in routine to compute this), what do you observe qualitatively and quantitatively?

In [ ]:
def load_image(filename):
  A = np.loadtxt(open(filename, "rb"), delimiter=",", skiprows=1)
  return A

def als_svd_approximations(filename):
  A = load_image(filename)

  # Our approx
  A_apprx = alternating_ls(A, k=75)

  # SVD
  U, S, Vt = np.linalg.svd(A, full_matrices=False)
  k = 75
  U_k = U[:, :k]
  S_k = S[:k]
  Vt_k = Vt[:k, :]
  A_svd = U_k @ np.diag(S_k) @ Vt_k

  # Calculate the Frobenius difference
  frob_diff = np.linalg.norm(A_apprx - A_svd, "fro")
  print("The Frobenius Difference is: ", frob_diff)

  return A_apprx, A_svd

A_approx, A_svd = als_svd_approximations("data/Cornell.csv")
  

In [ ]:
def compare_images(A, B, A_title, B_title)
  fig, axes = plt.subplots(1, 2, figsize=(18, 6))  # 1 row, 3 columns

  axes[0].imshow(A, cmap='gray')
  axes[0].set_title(A_title)
  axes[0].axis('on')

  axes[1].imshow(B, cmap='gray')
  axes[1].set_title(B_title)
  axes[1].axis('on')
  
  # Adjust layout to prevent overlap
  plt.tight_layout()

  # Show the combined figure
  plt.show()

compare_images(A_approx, A_svd, "ALS Rank-75 Approximation", 'Rank-75 SVD Reconstruction')
  

[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]]


## Question 1:
Show that if we want to solve $$\min_x \|Ax-b\|_2^2 + \beta^2 \|x\|_2^2$$ we can instead solve 
$$
\min_x \left\|\begin{bmatrix}A \\ \beta I\end{bmatrix}x-\begin{bmatrix}b \\ 0\end{bmatrix}\right\|_2^2.
$$

## Goal 4:
Using your $QR$ factorization, implement Algorithm 2. From the first set of goals, you should have worked out how to split this up into solving many least squares problems. Now, you have to think about what size those problems are, and which entries of the matrices they involve. Given that, you can then leverage your $QR$ factorization and the preceding item to implement the algorithm. **Please explain how you do this in the project report.**

## Goal 5:
Using your implementation, load each of the image files in Image*.csv and associated masks Mask*.csv, where * is 1, 2, and 3. Each file contains an image and the associated mask file contains an image of the same size whose entries are 1 if the corresponding entry of $C$ is observed and 0 if it is unobserved (so it defines the set $\Omega$). The unknown entries of $C$ have been set arbitrarily and you should not access or use them (as the will adversely affect your results). Using Algorithm 2, try and recover each of the underlying images. You now have some parameters to consider and we encourage exploration of their values. You should not have to go to $k$ larger than 75 for any of the examples, and good $\beta$ to try are between $10^{-2}$ and 1. Report the images you recover, and discuss what you observe.